In [2]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(1,'/Users/sadmin/Documents/MeerFish') # path to MeerFish code
import cosmo
import survey
import model
import fisher

f_tobsloss = 0.5

theta_ids = [\
    #r'$\overline{T}_{\rm HI}$',\
    #r'$b_1$',\
    #r'$b_2$',\
    #r'$b^\phi_1$',\
    #r'$b^\phi_2$',\
    #r'$f$',\
    #r'$\alpha_\perp$',\
    #r'$\alpha_\parallel$',\
    r'$A_{\rm BAO}$',\
    #r'$f_{\rm NL}$'\
    ]

Survey1_arg = 'MK_UHF'
Survey2_arg = 'DESI_LRG'
A_sky = 3000
t_obs = 800

In [3]:
zs = [0.65,0.8]

for i,z_i in enumerate(zs):
    zminzmax = [z_i-0.05,z_i+0.05] # use default
    z,zmin1,zmin2,zmax1,zmax2,A_sky1,A_sky2,V_bin1,V_bin2,V_binX,theta_FWHM1,theta_FWHM2,t_tot,N_dish,sigma_z1,sigma_z2,P_N,nbar = survey.params(Survey1=Survey1_arg,Survey2=Survey2_arg,zminzmax=zminzmax,A_sky1=A_sky,t_tot=t_obs)
    surveypars = z,V_bin1,V_bin2,V_binX,theta_FWHM1,theta_FWHM2,sigma_z1,sigma_z2,P_N,1/nbar
    cosmopars = cosmo.SetCosmology(z=z,return_cosmopars=True) # set initial default cosmology
    nuispars = 0,0,0
    Pmod = cosmo.MatterPk(z)

    #k,kbins,kmin,kmax = model.get_kbins(z,zmin1,zmax1,A_sky1)
    #k = np.arange(0.02,0.32,np.mean(np.diff(k)))
    k = np.linspace(0.02,0.32,50)
    '''
    ell = 0

    P_X = model.P_ell(ell,k,Pmod,cosmopars,surveypars,nuispars,'X')
    P_err = model.sigma_ell_error(ell,k,Pmod,cosmopars,surveypars,nuispars,'X')
    P_smooth,f_BAO_HI_wb = model.Pk_noBAO(P_X,k)
    plt.axhline(1,color='black',ls='--')
    plt.errorbar(k,P_X/P_smooth,P_err/P_smooth)
    #plt.loglog()
    plt.figure()
    #exit()
    '''

    #ells = [0,2,4]
    ells = [0]

    theta = model.get_param_vals(theta_ids,z,cosmopars)
    F = fisher.Matrix_ell(theta_ids,k,Pmod,cosmopars,surveypars,nuispars,ells,tracer='1')
    C = fisher.FisherInverse(F)
    SNR_BAO = 1/np.sqrt(C[0][0])
    print('HI auto ( z =',z,'):',SNR_BAO)

    F = fisher.Matrix_ell(theta_ids,k,Pmod,cosmopars,surveypars,nuispars,ells,tracer='X')
    C = fisher.FisherInverse(F)
    SNR_BAO = 1/np.sqrt(C[0][0])
    print('Cross ( z =',z,'):',SNR_BAO)


HI auto ( z = 0.65 ): 3.185834763434501
Cross ( z = 0.65 ): 4.599662648480008
HI auto ( z = 0.8 ): 2.7116633921662023
Cross ( z = 0.8 ): 4.292405560305796
